In [1]:
pip install langchain langgraph openai tavily-python wikipedia langchain-community python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal, List, Union
import operator

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import Runnable
from langchain_core.tools import Tool

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

from langchain.chat_models import AzureChatOpenAI
from langgraph.graph import END, StateGraph


In [3]:
load_dotenv()

True

In [4]:
llm = AzureChatOpenAI(
    deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["OPENAI_API_VERSION"],
    openai_api_key=os.environ["AZURE_OPENAI_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)


C:\Users\rishi\AppData\Local\Temp\ipykernel_21156\3388569776.py:1: LangChainDeprecationWarning: The class `AzureChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import AzureChatOpenAI``.
  llm = AzureChatOpenAI(


In [5]:
tavily_tool = TavilySearchResults(max_results=3)
wikipedia_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools = [tavily_tool, wikipedia_tool]


In [6]:
tool_selector_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Decide which tool is best to answer the question."),
    MessagesPlaceholder("messages")
])

In [7]:
class AgentState(TypedDict):
    messages: Annotated[List[Union[HumanMessage, AIMessage, ToolMessage]], operator.add]
    tool_choice: Union[Literal["tavily"], Literal["wikipedia"], None]
    tool_output: Union[str, None]

In [8]:
def decide_tool(state: AgentState) -> AgentState:
    question = state["messages"][-1].content
    if "recent" in question.lower() or "latest" in question.lower():
        return {**state, "tool_choice": "tavily"}
    else:
        return {**state, "tool_choice": "wikipedia"}

In [9]:
# --- 🔧 Node: Run Tool ---
def run_tool(state: AgentState) -> AgentState:
    query = state["messages"][-1].content
    if state["tool_choice"] == "tavily":
        result = tavily_tool.run(query)  # Tavily accepts raw string
    else:
        result = wikipedia_tool.run({"query": query})  # ✅ FIX: wrap in dict
    return {**state, "tool_output": result}


In [10]:
def generate_response(state):
    prompt = [
        SystemMessage(content="You are a helpful assistant. Use the following tool result to answer the user."),
        HumanMessage(content=state["messages"][-1].content),
        SystemMessage(content=f"Tool Result: {state['tool_output']}")
    ]
    response = llm.invoke(prompt)
    return {**state, "messages": state["messages"] + [response]}


In [11]:
graph_builder = StateGraph(AgentState)
graph_builder.add_node("choose_tool", decide_tool)
graph_builder.add_node("run_tool", run_tool)
graph_builder.add_node("respond", generate_response)

graph_builder.set_entry_point("choose_tool")
graph_builder.add_edge("choose_tool", "run_tool")
graph_builder.add_edge("run_tool", "respond")
graph_builder.set_finish_point("respond")

research_graph = graph_builder.compile()

In [12]:
initial_state = {
    "messages": [HumanMessage(content="What is the latest news about SpaceX?")],
    "tool_choice": None,
    "tool_output": None
}

final_state = research_graph.invoke(initial_state)
print("\n🔚 Final Response:")
print(final_state["messages"][-1].content)


🔚 Final Response:
Recently, SpaceX has been in the news for several reasons:

1. **Starship Test Delay**: There have been delays in the test flights of SpaceX's mega rocket, Starship, attributed to last-minute issues. This has affected Elon Musk's ambitions for Mars exploration.

2. **Crewed Missions**: SpaceX is set to fly three additional private crew missions aboard its Dragon spacecraft to the International Space Station (ISS) later this year, as announced by Axiom Space.

3. **Falcon 9 Milestone**: SpaceX achieved a new reusability milestone with its Falcon 9 rocket, further demonstrating its capabilities in launching payloads to space.

4. **Astronaut Rescues**: SpaceX has been involved in launching new crews to the ISS to replace NASA astronauts who faced issues, including a recent launch that was delayed due to a launch pad problem.

For more details, you can check out the latest updates on [SpaceX's official site](https://www.spacex.com/updates/) or follow the ongoing discuss